In [18]:

# ============================================================
# CELL 1: CONNECT GOOGLE DRIVE
# ============================================================

# Import Google Colab Drive utilities.
from google.colab import drive

# Mount Google Drive to access saved project files.
drive.mount('/content/drive')

# Import libraries required for the project.
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

print("Google Drive connected successfully!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive connected successfully!


In [19]:

# ============================================================
# CELL 2: DEFINE PROJECT FOLDERS
# ============================================================

# Define the main project directory in Google Drive.
PROJECT_PATH = "/content/drive/MyDrive/AMEX_Credit_Risk_Capstone_Project"

# Define the folders containing processed data, models and results.
DATA_PATH = os.path.join(PROJECT_PATH, "data", "processed")
MODELS_PATH = os.path.join(PROJECT_PATH, "models")
FIGURES_PATH = os.path.join(PROJECT_PATH, "results", "figures")
TABLES_PATH = os.path.join(PROJECT_PATH, "results", "tables")

# Define the SHAP results folders.
SHAP_PATH = os.path.join(FIGURES_PATH, "shap")
SHAP_RESULTS_PATH = os.path.join(TABLES_PATH, "shap")

# Create SHAP folders if they do not already exist.
os.makedirs(SHAP_PATH, exist_ok=True)
os.makedirs(SHAP_RESULTS_PATH, exist_ok=True)

# Verify that the project directory exists.
print("Project directory exists:", os.path.exists(PROJECT_PATH))
print("Models directory exists:", os.path.exists(MODELS_PATH))
print("Data directory exists:", os.path.exists(DATA_PATH))


Project directory exists: True
Models directory exists: True
Data directory exists: True


In [20]:

# ============================================================
# CELL 3: LOAD PREVIOUSLY TRAINED MODELS
# ============================================================

# Load the trained XGBoost model.
xgb_model = joblib.load(
    os.path.join(MODELS_PATH, "xgboost_model.pkl")
)

# Load the previously fitted median imputer.
median_imputer = joblib.load(
    os.path.join(MODELS_PATH, "median_imputer.pkl")
)

# Load the previously fitted standard scaler.
standard_scaler = joblib.load(
    os.path.join(MODELS_PATH, "standard_scaler.pkl")
)

# Display confirmation.
print("XGBoost model loaded successfully!")
print("Median imputer loaded successfully!")
print("Standard scaler loaded successfully!")


XGBoost model loaded successfully!
Median imputer loaded successfully!
Standard scaler loaded successfully!


In [21]:


# ============================================================
# CELL 4: LOAD VALIDATION DATA AND RESTORE FEATURE ORDER
# ============================================================

# Define the exact 30 features used during model development.
# Keep this order unchanged because the model expects these columns.
selected_features = [
    "P_2_last",
    "D_48_last",
    "B_2_last",
    "D_61_last",
    "B_9_last",
    "B_18_last",
    "D_55_last",
    "D_61_mean",
    "B_2_mean",
    "D_44_last",
    "B_9_mean",
    "R_1_mean",
    "B_3_last",
    "D_75_last",
    "B_7_last",
    "R_1_last",
    "B_4_last",
    "B_20_last",
    "B_19_last",
    "B_38_last",
    "B_1_last",
    "B_23_mean",
    "R_2_last",
    "B_22_mean",
    "R_10_mean",
    "B_17_mean",
    "B_38_mean",
    "B_22_last",
    "B_1_mean",
    "B_30_last"
]

# Load the customer-level validation dataset.
validation_modeling_data_15k = pd.read_csv(
    os.path.join(DATA_PATH, "validation_modeling_data_15k.csv")
)

# Extract customer IDs in their original order.
validation_customer_ids = (
    validation_modeling_data_15k["customer_ID"]
    .reset_index(drop=True)
)

# Extract the 30 model features in the specified order.
X_validation = validation_modeling_data_15k[
    selected_features
].copy()

# Extract the actual target values.
y_validation = validation_modeling_data_15k["target"].copy()

# Apply the previously fitted median imputer.
# Do not fit a new imputer on validation data.
X_validation_imputed = pd.DataFrame(
    median_imputer.transform(X_validation),
    columns=selected_features,
    index=X_validation.index
)

# Display dataset information.
print("Validation customers:", len(validation_customer_ids))
print("Selected features:", len(selected_features))
print("Imputed feature shape:", X_validation_imputed.shape)
print(
    "Missing values after imputation:",
    X_validation_imputed.isna().sum().sum()
)

# Verify that the model contains the expected number of features.
print(
    "Model feature count:",
    xgb_model.get_booster().num_features()
)



Validation customers: 15000
Selected features: 30
Imputed feature shape: (15000, 30)
Missing values after imputation: 0
Model feature count: 30


In [22]:

# ============================================================
# CELL 5: INITIALISE SHAP EXPLAINER
# ============================================================

# Create a TreeExplainer using the saved XGBoost model.
explainer = shap.TreeExplainer(xgb_model)

# Display confirmation.
print("SHAP TreeExplainer initialised successfully!")


SHAP TreeExplainer initialised successfully!


In [23]:

# ============================================================
# CELL 6: LOAD PREVIOUS INDIVIDUAL CUSTOMER RESULTS
# ============================================================

# Load the saved individual customer SHAP contributions.
customer_shap_df = pd.read_csv(
    os.path.join(
        SHAP_RESULTS_PATH,
        "individual_customer_SHAP_explanation.csv"
    )
)

# Load the saved individual customer prediction.
customer_prediction = pd.read_csv(
    os.path.join(
        SHAP_RESULTS_PATH,
        "individual_customer_prediction.csv"
    )
)

# Display the saved prediction.
display(customer_prediction)

# Display the customer's top SHAP contributions.
display(customer_shap_df.head(10))


,customer_ID,actual_target,predicted_class,default_probability,non_default_probability
0,00dae539b315802284dd4ea601580ff74d537a36e224de...,0,0,0.001625,0.998375


,Feature,SHAP_Value,Absolute_SHAP,Direction
0,P_2_last,-2.303136,2.303136,Reduces default output
1,B_2_mean,-0.492365,0.492365,Reduces default output
2,D_48_last,-0.424289,0.424289,Reduces default output
3,B_1_last,-0.388367,0.388367,Reduces default output
4,B_7_last,-0.306905,0.306905,Reduces default output
5,D_61_mean,-0.249551,0.249551,Reduces default output
6,B_9_last,-0.206185,0.206185,Reduces default output
7,D_75_last,-0.205555,0.205555,Reduces default output
8,B_3_last,-0.200294,0.200294,Reduces default output
9,R_1_mean,-0.164426,0.164426,Reduces default output


In [24]:

# Load the original 70K customer modelling dataset
model_data = pd.read_csv(
    os.path.join(DATA_PATH, "modeling_data_30_features_70k.csv")
)

# Extract the 30 model features in their original training order
selected_features = [
    col for col in model_data.columns
    if col not in ["customer_ID", "target"]
]

# Display the feature information
print("Number of selected features:", len(selected_features))
print("\nSelected features:")
print(selected_features)


Number of selected features: 30

Selected features:
['P_2_last', 'D_48_last', 'B_2_last', 'D_61_last', 'B_9_last', 'B_18_last', 'D_55_last', 'D_61_mean', 'B_2_mean', 'D_44_last', 'B_9_mean', 'R_1_mean', 'B_3_last', 'D_75_last', 'B_7_last', 'R_1_last', 'B_4_last', 'B_20_last', 'B_19_last', 'B_38_last', 'B_1_last', 'B_23_mean', 'R_2_last', 'B_22_mean', 'R_10_mean', 'B_17_mean', 'B_38_mean', 'B_22_last', 'B_1_mean', 'B_30_last']


In [25]:

# ============================================================
# CELL 5: RECREATE SHAP EXPLAINER
# ============================================================

# Import SHAP library
import shap

# Create a TreeExplainer using the saved XGBoost model
explainer = shap.TreeExplainer(xgb_model)

# Display confirmation
print("SHAP explainer created successfully!")

# Display the expected model output (base value)
print("Expected value:", explainer.expected_value)


SHAP explainer created successfully!
Expected value: [-1.05154479]


In [26]:

# ============================================================
# CELL 6: CUSTOMER-LEVEL DEFAULT RISK EXPLANATION FUNCTION
# ============================================================

def explain_customer(customer_id):

    # --------------------------------------------------------
    # 1. Search for the entered customer ID
    # --------------------------------------------------------

    customer_id = str(customer_id)

    matching_rows = validation_modeling_data_15k[
        validation_modeling_data_15k["customer_ID"].astype(str)
        == customer_id
    ]

    # Check whether the customer exists in the validation dataset
    if matching_rows.empty:
        print("Customer ID not found in the validation dataset.")
        return None

    # Extract the customer's 30 original features
    customer_raw = matching_rows[selected_features].copy()

    # --------------------------------------------------------
    # 2. Apply the previously fitted median imputer
    # --------------------------------------------------------

    # Transform missing values using the existing imputer
    customer_features = pd.DataFrame(
        median_imputer.transform(customer_raw),
        columns=selected_features
    )

    # --------------------------------------------------------
    # 3. Generate the XGBoost prediction
    # --------------------------------------------------------

    # Calculate the probability of each class
    probabilities = xgb_model.predict_proba(customer_features)[0]

    # Extract the default and non-default probabilities
    default_probability = probabilities[1]
    non_default_probability = probabilities[0]

    # Predict the class using a 0.5 threshold
    predicted_class = int(default_probability >= 0.5)

    # --------------------------------------------------------
    # 4. Calculate individual SHAP contributions
    # --------------------------------------------------------

    # Explain this customer's prediction
    customer_shap_values = explainer(customer_features)

    # Extract the SHAP values for the customer
    contributions = customer_shap_values.values[0]

    # Create a DataFrame containing feature contributions
    shap_df = pd.DataFrame({
        "Feature": selected_features,
        "SHAP_Value": contributions
    })

    # --------------------------------------------------------
    # 5. Identify the strongest risk contributors
    # --------------------------------------------------------

    # Positive SHAP values push the prediction towards default
    top_default_factors = (
        shap_df[shap_df["SHAP_Value"] > 0]
        .sort_values("SHAP_Value", ascending=False)
        .head(5)
    )

    # Negative SHAP values push the prediction away from default
    top_non_default_factors = (
        shap_df[shap_df["SHAP_Value"] < 0]
        .sort_values("SHAP_Value", ascending=True)
        .head(5)
    )

    # --------------------------------------------------------
    # 6. Display the prediction summary
    # --------------------------------------------------------

    print("=" * 60)
    print("CUSTOMER CREDIT DEFAULT RISK EXPLANATION")
    print("=" * 60)

    print("\nCustomer ID:", customer_id)
    print(f"Default Probability: {default_probability:.2%}")
    print(f"Non-default Probability: {non_default_probability:.2%}")

    print(
        "Predicted Class:",
        "DEFAULT" if predicted_class == 1 else "NON-DEFAULT"
    )

    # --------------------------------------------------------
    # 7. Display factors increasing default risk
    # --------------------------------------------------------

    print("\nTOP FACTORS INCREASING DEFAULT RISK")
    print("-" * 60)

    if not top_default_factors.empty:
        display(top_default_factors.reset_index(drop=True))
    else:
        print("No positive SHAP contributions.")

    # --------------------------------------------------------
    # 8. Display factors reducing default risk
    # --------------------------------------------------------

    print("\nTOP FACTORS REDUCING DEFAULT RISK")
    print("-" * 60)

    if not top_non_default_factors.empty:
        display(top_non_default_factors.reset_index(drop=True))
    else:
        print("No negative SHAP contributions.")

    # --------------------------------------------------------
    # 9. Generate the individual SHAP waterfall plot
    # --------------------------------------------------------

    print("\nINDIVIDUAL CUSTOMER EXPLANATION")

    shap.plots.waterfall(
        customer_shap_values[0],
        max_display=15
    )

    # --------------------------------------------------------
    # 10. Return the results for future interface development
    # --------------------------------------------------------

    return {
        "customer_id": customer_id,
        "default_probability": default_probability,
        "non_default_probability": non_default_probability,
        "predicted_class": predicted_class,
        "default_factors": top_default_factors,
        "non_default_factors": top_non_default_factors,
        "shap_values": customer_shap_values
    }


# Confirm that the function has been created
print("Customer explanation function created successfully!")


Customer explanation function created successfully!


In [27]:

# ============================================================
# CELL 7: TEST CUSTOMER-LEVEL EXPLANATION
# ============================================================

# Enter a customer ID from the validation dataset
customer_id_input = input("Enter Customer ID: ")

# Generate the prediction and SHAP explanation
customer_result = explain_customer(customer_id_input)


KeyboardInterrupt: Interrupted by user

In [28]:
# Import Google Drive utilities
from google.colab import drive
import os

# Mount Google Drive (skip this if it is already mounted)
drive.mount('/content/drive')

# Define the project models folder
MODELS_PATH = "/content/drive/MyDrive/AMEX_Credit_Risk_Capstone_Project/models"

# Export the already-trained XGBoost model in JSON format
xgb_model.save_model(
    os.path.join(MODELS_PATH, "xgboost_model.json")
)

print("XGBoost model exported successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
XGBoost model exported successfully!
